# SYSCOHADA Pipeline - Plan de correction et execution distante
Ce notebook documente et prepare la migration du pipeline vers une execution sur Google Colab ou Kaggle.

La machine locale ne lance pas l'inference OCR/VLM. Elle sert uniquement aux controles statiques et aux tests legers.

## Objectifs
1. Stabiliser le contrat d'execution et les formats PaddleOCR.
2. Brancher les checkpoints et la reprise par document/page.
3. Completer les regles comptables et le consensus.
4. Produire des sorties Excel auditees.
5. Mesurer les resultats uniquement sur une machine distante equipee pour l'OCR/VLM.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
PACKAGE_ROOT = PROJECT_ROOT / 'syscohada_pipeline'
print('Projet :', PROJECT_ROOT)
print('Package present :', PACKAGE_ROOT.exists())
print('Execution distante attendue :', 'Colab' if 'COLAB_RELEASE_TAG' in os.environ else 'Kaggle' if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ else 'Environnement local')

## Installation distante
Executer cette cellule uniquement sur Colab ou Kaggle. Adapter la version CUDA a l'image disponible.

In [ ]:
# Colab/Kaggle uniquement
# !apt-get update -qq && apt-get install -y -qq poppler-utils
# !pip install -q -e .
# !pip install -q pytest

## Controle structurel sans inference
Cette cellule ne charge aucun modele et peut etre executee localement ou a distance.

In [ ]:
import ast

def python_files(root):
    return sorted(root.rglob('*.py'))

def function_lengths(path):
    tree = ast.parse(path.read_text(encoding='utf-8'))
    return [(node.name, node.end_lineno - node.lineno + 1) for node in ast.walk(tree) if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]

violations = []
for path in python_files(PACKAGE_ROOT):
    for name, length in function_lengths(path):
        if length > 25:
            violations.append((str(path), name, length))
print('Fonctions > 25 lignes :', len(violations))
for violation in violations:
    print(violation)

## Installation et tests legers
Avant toute inference, corriger les imports de tests, installer Pytest et verifier que les moteurs ne sont pas initialises pendant la collecte.

In [ ]:
# A executer dans le terminal Colab/Kaggle ou dans une cellule shell distante
# !python -m compileall -q syscohada_pipeline
# !python -m pytest -q syscohada_pipeline/tests

## Ordre d'integration recommande
- Phase 0 : point d'entree unique, CLI, configuration et dependances de test.
- Phase 1 : schemas communs pour pages, cellules, audits et erreurs.
- Phase 2 : checkpoints atomiques, reprise et isolation de chaque document.
- Phase 3 : adaptateurs PaddleOCR et classification robuste.
- Phase 4 : validations Bilan, TFT et reconciliation interannuelle.
- Phase 5 : consensus reel et confiance par cellule.
- Phase 6 : sorties Excel, metriques et non-regression.

## Execution des strategies
Les commandes suivantes sont a lancer sur Colab/Kaggle apres installation et depot des PDF. Elles ne doivent pas etre lancees sur la machine locale.

In [ ]:
# Exemple de traitement distant d'un document ou d'un dossier
INPUT = 'data/input'
OUTPUT = 'data/output'

# !python syscohada_pipeline/main_a.py $INPUT $OUTPUT
# !python syscohada_pipeline/main_b.py $INPUT $OUTPUT
# !python syscohada_pipeline/main_c.py $INPUT $OUTPUT
# !python syscohada_pipeline/main_d.py $INPUT $OUTPUT

## Criteres d'acceptation
- Aucun traitement ne demarre si le contrat de configuration est invalide.
- Une fonction de production reste sous 25 lignes.
- Un document echoue est journalise et n'interrompt pas le lot.
- Une reprise utilise le dernier checkpoint valide.
- Une valeur absente n'est jamais convertie implicitement en zero.
- Les controles comptables apparaissent dans la feuille d'audit.
- Les conflits entre moteurs sont conserves et ne sont pas masques.
- Les metriques finales sont produites sur Colab/Kaggle avec les memes fixtures de validation.

## Livrables de la branche
- `README.md` : documentation specifique du package et plan de correction.
- `syscohada_pipeline_plan.ipynb` : preparation des controles et de l'execution distante.
Aucun autre notebook existant n'est modifie.